# Earthquake LSTM Workflow: Steps 1-8

This notebook runs the preprocessing and feature-engineering steps one step at a time. It is self-contained, so it can run without importing `method.py`. Each step saves a CSV file in `data/step_outputs` so the intermediate changes can be checked.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

DATA_PATH = Path("data") / r"earthquake_1995-2023.csv"
STEP_OUTPUT_DIR = Path("data") / "step_outputs"
REQUIRED_COLUMNS = ["date_time", "magnitude", "depth", "tsunami", "latitude", "longitude"]
DATE_FORMAT = "%d-%m-%Y %H:%M"
STEP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)

def save_step(df: pd.DataFrame, filename: str) -> Path:
    output_path = STEP_OUTPUT_DIR / filename
    if isinstance(df.index, pd.RangeIndex):
        df.to_csv(output_path, index=False)
    else:
        df.reset_index().to_csv(output_path, index=False)
    print(f"Saved: {output_path}")
    return output_path

def show_step_output(df: pd.DataFrame, preview_rows: int = 5) -> None:
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns):,}")
    display(df.head(preview_rows))
    display(df.tail(preview_rows))

## Step 1: Load Main Dataset

In [ ]:
# CREATE def of STEP 1 BELOW HERE:


step_1_df = s1_load_main_dataset(DATA_PATH)

save_step(step_1_df, "step_1_loaded_dataset.csv")
print(f"Input file: {DATA_PATH}")
print("Missing values in required columns:")
display(step_1_df[["date_time", "magnitude", "depth", "tsunami", "latitude", "longitude"]].isna().sum().to_frame("missing_count"))
show_step_output(step_1_df)

## Step 2: Date-Time Parsing and Chronological Sorting

In [ ]:
before_rows = len(step_1_df)
invalid_dates = pd.to_datetime(
    step_1_df["date_time"],
    format="%d-%m-%Y %H:%M",
    errors="coerce",
).isna().sum()

# CREATE def of STEP 2 BELOW HERE:


step_2_df = s2_parse_datetime_and_sort(step_1_df)

save_step(step_2_df, "step_2_datetime_sorted.csv")
print(f"Rows before: {before_rows:,}")
print(f"Invalid date rows removed: {invalid_dates:,}")
print(f"Rows after: {len(step_2_df):,}")
print(f"Date range: {step_2_df['date_time'].min()} to {step_2_df['date_time'].max()}")
show_step_output(step_2_df[["date_time", "magnitude", "depth", "tsunami", "latitude", "longitude"]])

## Step 3: Conservative Event Deduplication

In [ ]:
before_rows = len(step_2_df)
# CREATE def of STEP 3 BELOW HERE:


step_3_df = s3_conservative_deduplication(step_2_df)
removed_rows = before_rows - len(step_3_df)

save_step(step_3_df, "step_3_deduplicated_events.csv")
print(f"Rows before: {before_rows:,}")
print(f"Duplicate rows removed: {removed_rows:,}")
print(f"Rows after: {len(step_3_df):,}")
show_step_output(step_3_df[["date_time", "magnitude", "depth", "tsunami", "latitude", "longitude"]])

## Step 4: Monthly Resampling and Aggregation

In [ ]:
# CREATE def of STEP 4 BELOW HERE:


step_4_df = s4_monthly_resampling_and_aggregation(step_3_df)

save_step(step_4_df, "step_4_monthly_aggregation.csv")
print(f"Event rows before aggregation: {len(step_3_df):,}")
print(f"Monthly rows after aggregation: {len(step_4_df):,}")
print(f"Monthly date range: {step_4_df.index.min()} to {step_4_df.index.max()}")
display(step_4_df.describe().T)
show_step_output(step_4_df)

## Step 5: Missing Month Handling

In [ ]:
expected_months = pd.date_range(step_4_df.index.min(), step_4_df.index.max(), freq="MS")
missing_months_before = expected_months.difference(step_4_df.index)

# CREATE def of STEP 5 BELOW HERE:


step_5_df = s5_missing_month_handling(step_4_df)
zero_event_months = (step_5_df["event_count"] == 0).sum()

save_step(step_5_df, "step_5_missing_months_filled.csv")
print(f"Missing calendar months before: {len(missing_months_before):,}")
print(f"Monthly rows before: {len(step_4_df):,}")
print(f"Monthly rows after: {len(step_5_df):,}")
print(f"Zero-event months after filling: {zero_event_months:,}")
if len(missing_months_before) > 0:
    display(pd.DataFrame({"missing_month_added": missing_months_before}))
show_step_output(step_5_df)

## Step 6: Missing Value Handling

In [ ]:
missing_before = step_5_df.isna().sum().to_frame("missing_before")

# CREATE def of STEP 6 BELOW HERE:


step_6_df = s6_missing_value_handling(step_5_df)
missing_after = step_6_df.isna().sum().to_frame("missing_after")
missing_report = missing_before.join(missing_after, how="outer").fillna(0).astype(int)
missing_flag_cols = [col for col in step_6_df.columns if col.endswith("_missing_flag")]

save_step(step_6_df, "step_6_missing_values_imputed.csv")
print("Missing indicator columns added:")
print(missing_flag_cols)
display(missing_report)
show_step_output(step_6_df)

## Step 7: Outlier Detection and Flagging

In [ ]:
# CREATE def of STEP 7 BELOW HERE:


step_7_df = s7_outlier_detection_and_flagging(step_6_df)
added_cols = [col for col in step_7_df.columns if col not in step_6_df.columns]
outlier_cols = [col for col in step_7_df.columns if col.endswith("outlier_flag")]
outlier_counts = step_7_df[outlier_cols].sum().to_frame("flagged_months")

save_step(step_7_df, "step_7_outlier_flags.csv")
print("Columns added:")
print(added_cols)
display(outlier_counts)
display(step_7_df.loc[step_7_df[outlier_cols].any(axis=1), ["event_count", "max_magnitude", "mean_depth", "event_count_rolling_z"] + outlier_cols].head(20))
show_step_output(step_7_df)

## Step 8: Feature Extraction

In [ ]:
# CREATE def of STEP 8 BELOW HERE:


step_8_df = s8_feature_extraction(step_7_df)
added_cols = [col for col in step_8_df.columns if col not in step_7_df.columns]
new_missing = step_8_df[added_cols].isna().sum().to_frame("missing_created_by_lag_or_rolling")

save_step(step_8_df, "step_8_engineered_features.csv")
print(f"Columns before: {len(step_7_df.columns):,}")
print(f"Columns after: {len(step_8_df.columns):,}")
print("Feature columns added:")
print(added_cols)
display(new_missing)
show_step_output(step_8_df)